# Task 1 - build the dataset

Set the nuScenes path in the next cell, then **Run All**. You get one merged YOLO
dataset (`0 cone, 1 barrier, 2 stop_sign`) and `configs/data.yaml`, ready for Task 2.

COCO stop signs are downloaded automatically. Re-running is safe and cheap.

In [ ]:
# ---- edit this ----
NUSCENES = r"C:\path\to\nuscenes-mini"   # dataroot, or any folder above it
PSEUDO_LABEL = True                       # label stop signs on nuScenes frames
DEVICE = ""                               # "" = auto, or "cpu", or "0"
# -------------------

In [ ]:
import os, sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(REPO)
sys.path.insert(0, str(REPO))

from common.env import find_nuscenes_root, writable_root

NUSCENES = find_nuscenes_root(NUSCENES)   # fails here if the path is wrong
OUT = writable_root() / "datasets/yolo_3class"
print("nuScenes:", NUSCENES)
print("output  :", OUT)

In [ ]:
# Choose the val scenes.
#
# The devkit's mini_val (scene-0103, scene-0916) contains no barriers at all, so
# barrier AP would be unmeasurable. Barriers are concentrated in very few scenes,
# so the split has to be picked from the data: hold out one cone scene and one
# barrier scene, keeping the single richest scene for training.
from collections import Counter, defaultdict

from nuscenes.nuscenes import NuScenes

CLS = {"movable_object.trafficcone": "cone", "movable_object.barrier": "barrier"}
nusc = NuScenes("v1.0-mini", dataroot=str(NUSCENES), verbose=False)

per_scene = defaultdict(Counter)
for scene in nusc.scene:
    token = scene["first_sample_token"]
    while token:
        sample = nusc.get("sample", token)
        for ann in sample["anns"]:
            name = nusc.get("sample_annotation", ann)["category_name"]
            if name in CLS:
                per_scene[scene["name"]][CLS[name]] += 1
        token = sample["next"]

for name in sorted(per_scene):
    c = per_scene[name]
    print(f"{name}   cone={c['cone']:4d}   barrier={c['barrier']:4d}")

by_cone = sorted(per_scene, key=lambda s: -per_scene[s]["cone"])
by_barrier = sorted(per_scene, key=lambda s: -per_scene[s]["barrier"])
keep = by_barrier[0]   # richest scene stays in train
val_cone = next(s for s in by_cone if s != keep and per_scene[s]["cone"])
val_barrier = next(
    s for s in by_barrier if s not in (keep, val_cone) and per_scene[s]["barrier"]
)
VAL_SCENES = [val_cone, val_barrier]
VAL_ARG = " ".join(VAL_SCENES)

print(f"\nval  : {VAL_ARG}")
print(f"train: everything else, including {keep}")

In [ ]:
# nuScenes -> YOLO labels, plus a meta/ sidecar per image with K, camera height,
# pitch and ground-truth depth. Task 3 needs those.
!python -u -m data.nuscenes_to_yolo --nuscenes-root "{NUSCENES}" --out "{OUT}" --val-scenes {VAL_ARG}

In [ ]:
# COCO 2017 stop signs as class 2. First run downloads ~600 MB, later runs skip it.
!python -u -m data.coco_stopsign --out "{OUT}"

In [ ]:
# nuScenes never annotates stop signs, so without this the merge teaches the
# model that every stop sign in a nuScenes frame is background.
DEVICE_ARG = f"--device {DEVICE}" if DEVICE else ""
if PSEUDO_LABEL:
    !python -u -m data.pseudo_label --out "{OUT}" --conf 0.5 --imgsz 960 --batch 16 {DEVICE_ARG}
else:
    print("skipped - stop sign AP will suffer")

In [ ]:
# Write configs/data.yaml and run the checks.
!python -u -m data.merge --out "{OUT}"
!python -u -m data.sanity_check --out "{OUT}" --draw 12

In [ ]:
# Eyeball it: drawn boxes carry the ground-truth depth, and the contact sheet
# shows what the pseudo-labeller called a stop sign.
import itertools

from IPython.display import Image, display

sheet = REPO / "results/pseudo_label_review.jpg"
if sheet.exists():
    display(Image(str(sheet)))
for p in itertools.islice(sorted((REPO / "results/sanity/val").glob("*.jpg")), 4):
    display(Image(str(p), width=900))

Done when `val:` shows all three classes non-zero and the checks pass. Then:

```
python -m eval.det_eval --run B0        # zero-shot floor
python -m detection.train --run B1      # training
```